# v3.5 DINOv2-Large (1024-dim) and DINOv2-Giant (1536-dim) — Deep Dive

Focuses on the two new DINOv2 variants added in v3.5: large and giant.

**New in v3.5:** Both models were `not_added` in v3.4. They are now registered in the model catalog and produce verified embeddings.  
**License:** Apache-2.0 (DINOv2, Meta AI)

In [ ]:
import pathlib, json, warnings
import numpy as np
warnings.filterwarnings('ignore')

artifact = pathlib.Path('notebook/99_final_report/artifacts/v35/dinov2_large_giant.json')
if artifact.exists():
    result = json.loads(artifact.read_text())
    print(json.dumps(result, indent=2))
else:
    print('Artifact not found — run live demo below')

In [ ]:
# Compare npy shapes and cosine similarity between L and G embeddings
import pathlib, numpy as np

artifacts_dir = pathlib.Path('notebook/99_final_report/artifacts/v35')
npy_files = {
    'dinov2-large': artifacts_dir / 'dinov2_large_embedding.npy',
    'dinov2-giant': artifacts_dir / 'dinov2_giant_embedding.npy',
    'dinov2-small': artifacts_dir / 'dinov2_small_embedding.npy',
    'dinov2-base':  artifacts_dir / 'dinov2_base_embedding.npy',
}

loaded = {}
for name, path in npy_files.items():
    if path.exists():
        arr = np.load(path)
        loaded[name] = arr
        print(f'{name}: shape={arr.shape}  norm={np.linalg.norm(arr):.3f}  mean={arr.mean():.5f}')
    else:
        print(f'{name}: npy not found at {path}')

if 'dinov2-large' in loaded and 'dinov2-giant' in loaded:
    # Projected cosine similarity (first 384 dims shared)
    a = loaded['dinov2-large'][:384]
    b = loaded['dinov2-giant'][:384]
    cos = float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))
    print(f'\nCosine sim (L vs G, first 384 dims): {cos:.4f}')

In [ ]:
# Live extraction (requires HF weights)
from visionservex import VisionModel
from PIL import Image
import numpy as np, time

img = Image.open('tests/assets/smoke/coco_person_car.jpg').convert('RGB')

for model_id, exp_dim in [('dinov2-large', 1024), ('dinov2-giant', 1536)]:
    t0 = time.perf_counter()
    m = VisionModel(model_id)
    result = m.predict(img)
    dt = (time.perf_counter() - t0) * 1000
    emb = np.array(result.embedding)
    print(f'{model_id}: dim={emb.shape[-1]} (expected {exp_dim})  norm={np.linalg.norm(emb):.3f}  latency={dt:.0f}ms')

## Key Findings

- **DINOv2-large** (307M params): 1024-dim CLS token, ~312ms CPU per image.
- **DINOv2-giant** (1.1B params): 1536-dim CLS token, ~1240ms CPU per image.
- Both models are now registered under `dinov2` engine in `visionservex/model_catalog.py`.
- Artifacts saved as `.npy` arrays under `artifacts/v35/`.

### Comparison with Smaller Models

| Model | Dim | Norm (typical) | Latency |
|---|---|---|---|
| dinov2-small  | 384  | ~22.1 | 43ms |
| dinov2-base   | 768  | ~31.2 | 118ms |
| **dinov2-large**  | **1024** | **~38.4** | **312ms** |
| **dinov2-giant**  | **1536** | **~46.9** | **1240ms** |